# ABAC Per-AU Data Quality Checks

## Purpose

Checks data availability **per Assessment Unit (AU)** for each ABAC metric.  
Unlike the table-level DQ check, this shows: *for each AU under each metric,  
what percentage of the data rows have valid (non-null, non-blank) values?*

## Column Classification

| IS_PRIMARY | Meaning | Used For |
|-----------|---------|----------|
| **Y** | Primary column — used in COUNT(DISTINCT) or final aggregation | Auditing (≥95% threshold) |
| **N** | Bridging/filter column — used for AU mapping or business logic filters | Context/debugging |

## Results Table

`RA_FY_2025.abac_dq_by_metric_au` — one row per (metric, AU, column)

---

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
# Hardcoded because %run doesn't work in .ipynb format.
# SNAPSHOT_CATALOGUE_NAME comes from the RA Settings config.
# ============================================================
SNAPSHOT_CATALOGUE_NAME = 'RA_FY_2025'

import datetime
from pytz import timezone

def getTodayDate():
    tz = timezone('US/Eastern')
    return datetime.datetime.now(tz).strftime('%Y-%m-%d')

today_date = getTodayDate()
TABLE_NAME = f'{SNAPSHOT_CATALOGUE_NAME}.abac_dq_by_metric_au'
print(f'Run Date: {today_date}')
print(f'Results Table: {TABLE_NAME}')

In [ ]:
# ============================================================
# CREATE PER-AU DQ RESULTS TABLE
# ============================================================
# Each row = one column check for one AU under one metric.
#
# IS_PRIMARY:
#   'Y' = Primary column (used in COUNT(DISTINCT) or aggregation)
#         → This is what Subham needs for auditing (≥95% threshold)
#   'N' = Bridging/filter column (used for AU mapping or business filters)
#         → Context only, not required for audit
# ============================================================
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
    METRIC_ID        STRING COMMENT 'Metric identifier (EBA01, TP01, etc.)',
    METRIC_DESC      STRING COMMENT 'Human-readable metric description',
    AU_ID            STRING COMMENT 'Assessment Unit BusinessID from fy25_list_of_units',
    AU_NAME          STRING COMMENT 'Assessment Unit name',
    DATA_ELEMENT     STRING COMMENT 'Column name being checked',
    IS_PRIMARY       STRING COMMENT 'Y=primary (COUNT DISTINCT col), N=bridging/filter',
    SOURCE           STRING COMMENT 'Data source (always ADIDO for ABAC)',
    SRC_TABLE_NAME   STRING COMMENT 'Source table(s) description',
    TOTAL_ROWS       BIGINT COMMENT 'Total rows mapped to this AU',
    VALID_ROWS       BIGINT COMMENT 'Rows with non-null non-blank value for this column',
    AVAILABILITY_PCT DOUBLE COMMENT 'Percentage: 100 * VALID_ROWS / TOTAL_ROWS',
    RUN_DATE         STRING COMMENT 'Date of execution'
)
''')
print('Table created/verified.')

In [ ]:
# ============================================================
# CREATE COST CENTER MAPPING BOOTSTRAP VIEW
# ============================================================
# Replicates the logic from 00_CC_Mapping_Setup.ipynb.
# This view maps Cost Centers to Assessment Units.
# Used by all CC-driven metrics (EBA, EMP, GEO).
# ============================================================
spark.sql('''
CREATE OR REPLACE TEMP VIEW vw_cost_center_mapping_bootstrap AS
WITH Base_Data AS (
    SELECT 
        CASE WHEN TRIM(CAST(CostCenterId AS STRING)) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CAST(CostCenterId AS STRING)), 4), 4, '0')
             ELSE UPPER(TRIM(CAST(CostCenterId AS STRING))) END AS Cost_Center_ID,
        TRIM(CAST(AssessableUnitID AS STRING)) AS Primary_AU_ID,
        TRIM(AssessableUnitName) AS Primary_AU_Name,
        TRIM(Segment) AS Primary_Segment,
        TRIM(AdditionalAssessableUnitIDandNameandSegment) AS Col_E,
        TRIM(AdditionalAUID) AS Col_F
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),
Branch_Primary AS (
    SELECT Cost_Center_ID, Primary_AU_ID AS AU_ID, Primary_AU_Name AS AU_Name,
           Primary_Segment AS Segment_Name
    FROM Base_Data
    WHERE Col_E = 'Yes' OR Col_E IS NULL OR Col_E = ''
),
Branch_Additional_Raw AS (
    SELECT Cost_Center_ID,
           CONCAT_WS(' ', COALESCE(Col_E, ''), COALESCE(Col_F, '')) AS Mashed_String
    FROM Base_Data
    WHERE Col_E != 'Yes' AND Col_E IS NOT NULL AND Col_E != ''
),
Extracted_Blocks AS (
    SELECT Cost_Center_ID,
           EXPLODE(regexp_extract_all(Mashed_String, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS Raw_Block
    FROM Branch_Additional_Raw WHERE Mashed_String != ''
),
Separated_ID_And_Remainder AS (
    SELECT Cost_Center_ID,
           TRIM(regexp_extract(Raw_Block, '^([0-9]{6})', 1)) AS AU_ID,
           TRIM(REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \\t-]*', '')) AS Remainder
    FROM Extracted_Blocks WHERE TRIM(Raw_Block) != ''
),
Parsed_Additionals AS (
    SELECT Cost_Center_ID, AU_ID,
           CASE WHEN Remainder LIKE '%-%'
                THEN TRIM(regexp_extract(Remainder, '^(.*)[ \\t]*-[ \\t]*[^-]+$', 1))
                ELSE Remainder END AS AU_Name,
           CASE WHEN Remainder LIKE '%-%'
                THEN TRIM(regexp_extract(Remainder, '.*[ \\t]*-[ \\t]*([^-]+)$', 1))
                ELSE '' END AS Segment_Name
    FROM Separated_ID_And_Remainder
),
Cleaned_Stack AS (
    SELECT DISTINCT Cost_Center_ID, AU_ID,
           TRIM(REGEXP_REPLACE(REGEXP_REPLACE(AU_Name, '^"|"$', ''), '[ ]+', ' ')) AS AU_Name,
           TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Segment_Name, '^"|"$', ''), '[ ]+', ' ')) AS Segment_Name
    FROM (
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Branch_Primary
        UNION
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Parsed_Additionals
    )
    WHERE AU_ID IS NOT NULL AND AU_ID != ''
      AND AU_Name IS NOT NULL AND TRIM(AU_Name) != ''
)
SELECT DISTINCT Cost_Center_ID, AU_ID,
       FIRST_VALUE(AU_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS AU_Name,
       FIRST_VALUE(Segment_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS Segment_Name
FROM Cleaned_Stack
''')
print('CC bootstrap view created.')

In [ ]:
# ============================================================
# SHARED HELPER VIEWS
# ============================================================

# Master AU list — the anchor for all output (ensures every AU appears)
spark.sql(f'''
CREATE OR REPLACE TEMP VIEW master_aus AS
SELECT 
    TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
    TRIM(Business_Operational_Unit_Name) AS AU_Name
FROM {DB}.fy25_list_of_units
WHERE BusinessID IS NOT NULL
  AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
  AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
''')

# CC mapping with smart-padded cost center IDs
spark.sql('''
CREATE OR REPLACE TEMP VIEW cc_mapping AS
SELECT DISTINCT 
    TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
    CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$'
         THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0')
         ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
FROM vw_cost_center_mapping_bootstrap
WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
''')

au_count = spark.sql('SELECT count(DISTINCT BusinessID) FROM master_aus').collect()[0][0]
print(f'Master AUs loaded: {au_count}')
cc_count = spark.sql('SELECT count(DISTINCT Mapped_CC) FROM cc_mapping').collect()[0][0]
print(f'CC mappings loaded: {cc_count} unique cost centers')

In [ ]:
def run_cc_metric_dq(metric_id, metric_desc, data_sql, columns, src_desc):
    '''
    Run per-AU DQ check for a CC-driven metric.

    Args:
        metric_id: e.g. 'EBA01'
        metric_desc: e.g. 'Travel, Lodging, and Entertainment'
        data_sql: SQL that returns Cleaned_CC + all columns to check
        columns: list of (col_name, is_primary) tuples
        src_desc: description of data source
    '''
    print(f'\n{"="*60}')
    print(f'{metric_id}: {metric_desc}')
    print(f'Source: {src_desc}')
    print(f'{"="*60}')
    for col_name, is_primary in columns:
        try:
            cast_col = f'cast({col_name} as STRING)'
            spark.sql(f'''
            INSERT INTO {TABLE_NAME}
            SELECT
                '{metric_id}', '{metric_desc}',
                mast.BusinessID, mast.AU_Name,
                '{col_name}', '{is_primary}', 'ADIDO', '{src_desc}',
                COALESCE(p.total_rows, 0),
                COALESCE(p.valid_rows, 0),
                CASE WHEN COALESCE(p.total_rows, 0) = 0 THEN NULL
                     ELSE p.availability_pct END,
                '{today_date}'
            FROM master_aus mast
            LEFT JOIN (
                SELECT
                    m.AU_ID,
                    count(1) AS total_rows,
                    count(CASE WHEN {cast_col} IS NOT NULL AND {cast_col} <> '' THEN 1 END) AS valid_rows,
                    round(100.0 * count(CASE WHEN {cast_col} IS NOT NULL AND {cast_col} <> '' THEN 1 END)
                          / count(1), 2) AS availability_pct
                FROM ({data_sql}) d
                JOIN cc_mapping m ON d.Cleaned_CC = m.Mapped_CC
                GROUP BY m.AU_ID
            ) p ON mast.BusinessID = p.AU_ID
            ''')
            label = 'PRIMARY' if is_primary == 'Y' else 'BRIDGING'
            print(f'  ✓ {col_name} [{label}]')
        except Exception as e:
            print(f'  ✗ {col_name}: ERROR — {str(e)}')

In [ ]:
# OPTIONAL: Uncomment to clear previous run results
# spark.sql(f"DELETE FROM {TABLE_NAME} WHERE RUN_DATE = '{today_date}'")
# print(f'Cleared results for {today_date}')

## CC-Driven Metrics

These metrics bridge data to AUs via **Cost Center mapping**:  
Data table → Smart-pad CC → JOIN `vw_cost_center_mapping_bootstrap` → GROUP BY AU_ID

---

In [ ]:
# ============================================================
# EBA01 — Travel, Lodging, and Entertainment
# ============================================================
# Data: Coupa expense reports (7 monthly files) + Finance GL (6 files)
# These two sources are INDEPENDENT — a transaction is in either
# Coupa OR Finance, never both. UNIONed into one combined result.
# Primary: CatCode_Int (the category code that determines if a
#          transaction matches the 29 target expense categories)
# Bridging: Cleaned_CC (cost center, used to map rows to AUs)
# ============================================================
eba01_sql = '''
    SELECT
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0')
             ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int
    FROM (
        SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    ) coupa
    WHERE Account LIKE '%-%-%'
    UNION ALL
    SELECT
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0')
             ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int
    FROM (
        SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
    ) finance
'''

run_cc_metric_dq(
    'EBA01', 'Travel, Lodging, and Entertainment',
    eba01_sql,
    [('CatCode_Int', 'Y'), ('Cleaned_CC', 'N')],
    'Coupa + Finance (combined)'
)

In [ ]:
# ============================================================
# EBA02 — Travel/Lodging/Entertainment (Public Officials)
# ============================================================
# Data: Coupa only (filters for PublicOfficial = 'Y'/'YES')
# Primary: PublicOfficial (determines if the expense involves a PO)
# ============================================================
eba02_sql = '''
    SELECT
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0')
             ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        PublicOfficial
    FROM (
        SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    ) coupa
    WHERE Account LIKE '%-%-%'
'''

run_cc_metric_dq(
    'EBA02', 'Travel/Lodging/Entertainment (Public Officials)',
    eba02_sql,
    [('PublicOfficial', 'Y'), ('Cleaned_CC', 'N')],
    'Coupa expense reports'
)

In [ ]:
# ============================================================
# EBA04 — Gifts/Travel/Lodging Non-POs > $250
# ============================================================
# Data: Coupa only
# Primary: Total (dollar amount, checked against $250 threshold)
# ============================================================
eba04_sql = '''
    SELECT
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0')
             ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        Total
    FROM (
        SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account, PublicOfficial, Total FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    ) coupa
    WHERE Account LIKE '%-%-%'
'''

run_cc_metric_dq(
    'EBA04', 'Gifts/Travel/Lodging Non-POs > $250',
    eba04_sql,
    [('Total', 'Y'), ('Cleaned_CC', 'N')],
    'Coupa expense reports'
)

In [ ]:
# ============================================================
# EBA06 — Donations
# ============================================================
# Data: Finance GL only (CatCodes 292, 427)
# Primary: CatCode (determines if transaction is a donation)
# ============================================================
eba06_sql = '''
    SELECT
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0')
             ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        CatCode
    FROM (
        SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
    ) finance
'''

run_cc_metric_dq(
    'EBA06', 'Donations',
    eba06_sql,
    [('CatCode', 'Y'), ('Cleaned_CC', 'N')],
    'Finance GL data'
)

In [ ]:
# ============================================================
# EBA07 — Marketing Expenses
# ============================================================
# Data: Finance GL only (same tables as EBA06, different CatCodes)
# Primary: CatCode
# ============================================================
# Reuses eba06_sql — same Finance data, different business filter
run_cc_metric_dq(
    'EBA07', 'Marketing Expenses',
    eba06_sql,
    [('CatCode', 'Y'), ('Cleaned_CC', 'N')],
    'Finance GL data'
)

In [ ]:
# ============================================================
# EMP01 — Canadian Officer / PEP Presence
# ============================================================
# Data: HR PEP list
# Primary: Region (filtered for 'Canada' to identify Canadian PEPs)
# Note: Costcenter uses SUBSTRING_INDEX to extract CC before space
# ============================================================
emp01_sql = '''
    SELECT
        CASE WHEN TRIM(SUBSTRING_INDEX(Costcenter, ' ', 1)) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(SUBSTRING_INDEX(Costcenter, ' ', 1)), 4), 4, '0')
             ELSE UPPER(TRIM(SUBSTRING_INDEX(Costcenter, ' ', 1))) END AS Cleaned_CC,
        Region
    FROM hive_metastore.ra_adido_2025.employee_pep_list_as_of_oct312025
'''

run_cc_metric_dq(
    'EMP01', 'Canadian Officer / PEP Presence',
    emp01_sql,
    [('Region', 'Y'), ('Cleaned_CC', 'N')],
    'Employee PEP list'
)

In [ ]:
# ============================================================
# EMP03 — Attestations (ABAC & CoC Training)
# ============================================================
# Data: ABAC training + Code of Conduct training (combined)
# Primary: CompletionStatus, CompletionDateUTC, DueDate
#   (metric = MIN(ABAC %, CoC %) where compliance = DateUTC <= DueDate)
# ============================================================
emp03_sql = '''
    SELECT
        CASE WHEN TRIM(CAST(DeptID AS STRING)) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CAST(DeptID AS STRING)), 4), 4, '0')
             ELSE UPPER(TRIM(CAST(DeptID AS STRING))) END AS Cleaned_CC,
        CompletionStatus, CompletionDateUTC, DueDate
    FROM hive_metastore.ra_adido_2025.fy25_mltf_sanctions_and_abac_11282025_all_employees
    UNION ALL
    SELECT
        CASE WHEN TRIM(CAST(DeptID AS STRING)) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CAST(DeptID AS STRING)), 4), 4, '0')
             ELSE UPPER(TRIM(CAST(DeptID AS STRING))) END AS Cleaned_CC,
        CompletionStatus, CompletionDateUTC, DueDate
    FROM hive_metastore.ra_adido_2025.td_code_of_conduct_and_ethics_110102024_10312025
'''

run_cc_metric_dq(
    'EMP03', 'Attestations (ABAC & CoC Training)',
    emp03_sql,
    [('CompletionStatus', 'Y'), ('CompletionDateUTC', 'Y'), ('DueDate', 'Y'), ('Cleaned_CC', 'N')],
    'ABAC + CoC training records'
)

In [ ]:
# ============================================================
# EMP05 — Contingent Workers in High-Risk Jurisdictions
# ============================================================
# Data: HR enterprise employee list
# Primary: WorkerType (filtered for '%contingent%'), Workcountry
#   (joined with country risk ratings for high-risk filter)
# ============================================================
emp05_sql = '''
    SELECT
        CASE WHEN TRIM(CAST(CostCenterID AS STRING)) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CAST(CostCenterID AS STRING)), 4), 4, '0')
             ELSE UPPER(TRIM(CAST(CostCenterID AS STRING))) END AS Cleaned_CC,
        WorkerType, Workcountry
    FROM hive_metastore.ra_adido_2025.hrbirai_17930_enterprise_list_as_of_103125_new_employees_101025_10312025
'''

run_cc_metric_dq(
    'EMP05', 'Contingent Workers in High-Risk Jurisdictions',
    emp05_sql,
    [('WorkerType', 'Y'), ('Workcountry', 'Y'), ('Cleaned_CC', 'N')],
    'HR enterprise employee list'
)

In [ ]:
# ============================================================
# GEO02 — Business Travel to High-Risk Jurisdictions
# ============================================================
# Data: AMEX GBT (has Numberoftrips) + CWT (1 row per trip)
# Primary: DestinationCountry (joined with country risk ratings)
#          Numberoftrips (AMEX only, CWT has NULL)
# ============================================================
geo02_sql = '''
    SELECT
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0')
             ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        DestinationCountry, cast(Numberoftrips as STRING) as Numberoftrips
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_amex_gbt_aug12_oct31_25
    UNION ALL
    SELECT
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0')
             ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        DestinationCountry, NULL as Numberoftrips
    FROM hive_metastore.ra_adido_2025.compliance_country_destination_report_cwt_nov24_aug11_25
'''

run_cc_metric_dq(
    'GEO02', 'Business Travel to High-Risk Jurisdictions',
    geo02_sql,
    [('DestinationCountry', 'Y'), ('Numberoftrips', 'Y'), ('Cleaned_CC', 'N')],
    'AMEX GBT + CWT travel data'
)

## TP-Driven Metrics (Third Party)

These metrics use a completely different AU mapping architecture:  
`abac_request_paul_v3` → Comma Exception Dictionary → LOB Explosion →  
LIKE match to `third_party_unit_mapping.TPRM_Units` → Dual-Bridge JOIN to Master AUs

All TP metrics use `COUNT(DISTINCT EngagementNumber)` as the primary aggregation.

---

In [ ]:
# ============================================================
# CREATE TP ENGAGEMENT-TO-AU MAPPING VIEW
# ============================================================
# Replicates the LOB explosion + TPRM mapping + dual-bridge join
# from the TP metric notebooks (tp01-tp05).
#
# COMMA EXCEPTION DICTIONARY:
# 6 LOB names contain commas that would break the split() logic:
#   1. CAD PB - RESL, Everyday Banking, Savings & Investing
#   2. CAD PB - Card Payments, Loyalty, & Personal Lending
#   3. Transformation, Enablement, & Customer Experience
#   4. TECE - Strategy, Change, & Operational Excellence
#   5. TECE - Enterprise Digital Strategy, Innovation, & Payments
#   6. P&T - Integrations, Data and Payments
# These are temporarily replaced with [COMMA] before splitting.
#
# DUAL-BRIDGE JOIN:
# The mapping table's Assessable_Unit_ID can contain either a
# numeric BusinessID or a string AU Name. The join checks both.
#
# NOTE: Uses concat('%', ..., '%') instead of LIKE '%' || ... || '%'
# to avoid string quoting issues in the notebook format.
# ============================================================
tp_sql = """
CREATE OR REPLACE TEMP VIEW tp_au_mapping AS
WITH Mapped_AUs AS (
    SELECT DISTINCT
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),
Third_Party_Data AS (
    SELECT
        tp.EngagementNumber,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(tp.owning_lob,
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(tp.lob_using,
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using,
        tp.location_of_tp,
        tp.country_prod_serv_originates,
        tp.Td_Countries_Impacted,
        tp.Contract_Amount,
        tp.KPI_Number,
        tp.final_rating,
        tp.primary_product_serv,
        tp.owning_lob,
        tp.lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
    WHERE tp.EngagementNumber IS NOT NULL
),
Flattened_LOBs AS (
    SELECT
        td.*,
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Third_Party_Data td
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)
SELECT DISTINCT
    mast.BusinessID AS AU_ID,
    mast.AU_Name,
    f.EngagementNumber,
    f.location_of_tp,
    f.country_prod_serv_originates,
    f.Td_Countries_Impacted,
    f.Contract_Amount,
    f.KPI_Number,
    f.final_rating,
    f.primary_product_serv,
    f.owning_lob,
    f.lob_using
FROM Flattened_LOBs f
INNER JOIN Mapped_AUs map
    ON UPPER(f.Expanded_LOB) LIKE concat('%', UPPER(map.TPRM_Units), '%')
INNER JOIN master_aus mast
    ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
    OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
"""

spark.sql(tp_sql)

tp_count = spark.sql('SELECT count(DISTINCT AU_ID) FROM tp_au_mapping').collect()[0][0]
eng_count = spark.sql('SELECT count(DISTINCT EngagementNumber) FROM tp_au_mapping').collect()[0][0]
print(f'TP mapping created: {tp_count} AUs, {eng_count} unique engagements')

In [ ]:
def run_tp_metric_dq(metric_id, metric_desc, columns, src_desc):
    '''
    Run per-AU DQ check for a TP-driven metric.
    Uses COUNT(DISTINCT EngagementNumber) pattern matching TP metrics.
    '''
    print(f'\n{"="*60}')
    print(f'{metric_id}: {metric_desc}')
    print(f'Source: {src_desc}')
    print(f'{"="*60}')
    for col_name, is_primary in columns:
        try:
            cast_col = f'cast({col_name} as STRING)'
            # Use COUNT(DISTINCT EngagementNumber) pattern for TP metrics
            spark.sql(f'''
            INSERT INTO {TABLE_NAME}
            SELECT
                '{metric_id}', '{metric_desc}',
                mast.BusinessID, mast.AU_Name,
                '{col_name}', '{is_primary}', 'ADIDO', '{src_desc}',
                COALESCE(p.total_engagements, 0),
                COALESCE(p.valid_engagements, 0),
                CASE WHEN COALESCE(p.total_engagements, 0) = 0 THEN NULL
                     ELSE p.availability_pct END,
                '{today_date}'
            FROM master_aus mast
            LEFT JOIN (
                SELECT
                    AU_ID,
                    count(DISTINCT EngagementNumber) AS total_engagements,
                    count(DISTINCT CASE WHEN {cast_col} IS NOT NULL AND {cast_col} <> ''
                          THEN EngagementNumber END) AS valid_engagements,
                    round(100.0 * count(DISTINCT CASE WHEN {cast_col} IS NOT NULL AND {cast_col} <> ''
                          THEN EngagementNumber END) / count(DISTINCT EngagementNumber), 2) AS availability_pct
                FROM tp_au_mapping
                GROUP BY AU_ID
            ) p ON mast.BusinessID = p.AU_ID
            ''')
            label = 'PRIMARY' if is_primary == 'Y' else 'BRIDGING'
            print(f'  ✓ {col_name} [{label}]')
        except Exception as e:
            print(f'  ✗ {col_name}: ERROR — {str(e)}')

In [ ]:
# ============================================================
# TP01 — Third Party High-Risk Jurisdictions
# Primary: EngagementNumber (COUNT DISTINCT)
# Filter: location_of_tp, country_prod_serv_originates,
#         Td_Countries_Impacted (country columns for risk check)
# ============================================================
run_tp_metric_dq(
    'TP01', 'TP High-Risk Jurisdictions',
    [('EngagementNumber', 'Y'),
     ('location_of_tp', 'N'),
     ('country_prod_serv_originates', 'N'),
     ('Td_Countries_Impacted', 'N'),
     ('owning_lob', 'N'), ('lob_using', 'N')],
    'abac_request_paul_v3'
)

# ============================================================
# TP02 — Third Party High Value (≥$1M)
# Primary: EngagementNumber, Contract_Amount
# ============================================================
run_tp_metric_dq(
    'TP02', 'TP High Value (>=1M)',
    [('EngagementNumber', 'Y'), ('Contract_Amount', 'Y'),
     ('owning_lob', 'N'), ('lob_using', 'N')],
    'abac_request_paul_v3'
)

# ============================================================
# TP03 — Third Party Services (Marketing/Brokerage)
# Primary: EngagementNumber
# Filter: KPI_Number, final_rating, primary_product_serv
# ============================================================
run_tp_metric_dq(
    'TP03', 'TP Services (Marketing/Brokerage)',
    [('EngagementNumber', 'Y'),
     ('KPI_Number', 'N'), ('final_rating', 'N'), ('primary_product_serv', 'N'),
     ('owning_lob', 'N'), ('lob_using', 'N')],
    'abac_request_paul_v3'
)

# ============================================================
# TP04 — Third Party Sole Source in High-Risk Jurisdictions
# Primary: EngagementNumber
# Filter: location columns (same as TP01)
# Note: Also depends on published_contrcats table (checked separately)
# ============================================================
run_tp_metric_dq(
    'TP04', 'TP Sole Source in High-Risk Jurisdictions',
    [('EngagementNumber', 'Y'),
     ('location_of_tp', 'N'),
     ('country_prod_serv_originates', 'N'),
     ('Td_Countries_Impacted', 'N'),
     ('owning_lob', 'N'), ('lob_using', 'N')],
    'abac_request_paul_v3'
)

# ============================================================
# TP05 — Third Party Government Interaction
# Primary: EngagementNumber
# Filter: KPI_Number, final_rating
# ============================================================
run_tp_metric_dq(
    'TP05', 'TP Government Interaction',
    [('EngagementNumber', 'Y'),
     ('KPI_Number', 'N'), ('final_rating', 'N'),
     ('owning_lob', 'N'), ('lob_using', 'N')],
    'abac_request_paul_v3'
)

print(f'\n{"="*60}')
print('ALL TP METRICS COMPLETE')
print(f'{"="*60}')

## Results

---

In [ ]:
# All per-AU DQ results
display(
    spark.sql(f'''
        SELECT * FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
        ORDER BY METRIC_ID, AU_ID, IS_PRIMARY DESC, DATA_ELEMENT
    ''')
)

In [ ]:
# ============================================================
# AUDIT FLAG: Primary columns below 95% threshold
# ============================================================
# These are the rows Subham needs for auditing.
# Primary columns (IS_PRIMARY = 'Y') with < 95% availability
# should be investigated.
# ============================================================
display(
    spark.sql(f'''
        SELECT
            METRIC_ID, AU_ID, AU_NAME, DATA_ELEMENT,
            TOTAL_ROWS, VALID_ROWS, AVAILABILITY_PCT,
            CASE WHEN AVAILABILITY_PCT IS NULL THEN 'NO DATA'
                 WHEN AVAILABILITY_PCT >= 95 THEN 'PASS'
                 WHEN AVAILABILITY_PCT >= 80 THEN 'WARN'
                 ELSE 'FAIL' END AS STATUS
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
          AND IS_PRIMARY = 'Y'
          AND (AVAILABILITY_PCT < 95 OR AVAILABILITY_PCT IS NULL)
        ORDER BY AVAILABILITY_PCT ASC NULLS FIRST
    ''')
)

In [ ]:
# ============================================================
# SUMMARY BY METRIC (Primary columns only)
# ============================================================
display(
    spark.sql(f'''
        SELECT
            METRIC_ID,
            METRIC_DESC,
            COUNT(DISTINCT AU_ID) AS num_aus,
            COUNT(DISTINCT DATA_ELEMENT) AS primary_cols_checked,
            ROUND(AVG(AVAILABILITY_PCT), 2) AS avg_availability,
            ROUND(MIN(AVAILABILITY_PCT), 2) AS min_availability,
            SUM(CASE WHEN AVAILABILITY_PCT >= 95 THEN 1 ELSE 0 END) AS pass_count,
            SUM(CASE WHEN AVAILABILITY_PCT < 95 OR AVAILABILITY_PCT IS NULL THEN 1 ELSE 0 END) AS flag_count
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
          AND IS_PRIMARY = 'Y'
        GROUP BY METRIC_ID, METRIC_DESC
        ORDER BY min_availability ASC NULLS FIRST
    ''')
)

In [ ]:
# ============================================================
# PER-AU HEALTH: Lowest primary column availability per AU
# ============================================================
# Shows each AU's worst primary column availability across all metrics.
# AUs with min < 95% need investigation.
# ============================================================
display(
    spark.sql(f'''
        SELECT
            AU_ID, AU_NAME,
            COUNT(DISTINCT METRIC_ID) AS metrics_covered,
            ROUND(AVG(AVAILABILITY_PCT), 2) AS avg_availability,
            ROUND(MIN(AVAILABILITY_PCT), 2) AS min_availability,
            SUM(CASE WHEN AVAILABILITY_PCT >= 95 THEN 1 ELSE 0 END) AS pass_count,
            SUM(CASE WHEN AVAILABILITY_PCT < 95 OR AVAILABILITY_PCT IS NULL THEN 1 ELSE 0 END) AS flag_count
        FROM {TABLE_NAME}
        WHERE RUN_DATE = '{today_date}'
          AND IS_PRIMARY = 'Y'
        GROUP BY AU_ID, AU_NAME
        ORDER BY min_availability ASC NULLS FIRST
    ''')
)